In [20]:
import pandas as pd

In [21]:
#Importar el archivo csv sin header
df = pd.read_csv('/content/orders_year.csv', header=None)
print(df.shape)
df.head()

(96688, 7)


,0,1,2,3,4,5,6
0,2022-06-10 7:22 pm,1110,Shirley Temple,NaN,2.0,10.0,NaN
1,2022-06-10 7:23 pm,1111,Freddie Mercury,NaN,1.0,25.0,NaN
2,2022-06-10 7:23 pm,1111,Michael Jackson,NaN,1.0,20.0,NaN
3,2022-06-10 7:26 pm,1112,Jarra Appleton,NaN,1.0,38.0,NaN
4,2022-06-10 7:26 pm,1112,Pilsen,NaN,1.0,10.0,NaN


In [22]:
#Nombrar las columnas
df.columns = ["Fecha", "Nro. del Pedido", "Nombre del Producto", "Variación del Producto", "Cantidad", "Precio", "Descuento" ]
df.head()

,Fecha,Nro. del Pedido,Nombre del Producto,Variación del Producto,Cantidad,Precio,Descuento
0,2022-06-10 7:22 pm,1110,Shirley Temple,NaN,2.0,10.0,NaN
1,2022-06-10 7:23 pm,1111,Freddie Mercury,NaN,1.0,25.0,NaN
2,2022-06-10 7:23 pm,1111,Michael Jackson,NaN,1.0,20.0,NaN
3,2022-06-10 7:26 pm,1112,Jarra Appleton,NaN,1.0,38.0,NaN
4,2022-06-10 7:26 pm,1112,Pilsen,NaN,1.0,10.0,NaN


In [23]:
#Verificar los data types
print(df.dtypes)
print('n/ Sumar Nulos')
print(df.isnull().sum())

Fecha                      object
Nro. del Pedido             int64
Nombre del Producto        object
Variación del Producto     object
Cantidad                  float64
Precio                    float64
Descuento                 float64
dtype: object
n/ Sumar Nulos
Fecha                         0
Nro. del Pedido               0
Nombre del Producto           1
Variación del Producto    77346
Cantidad                      1
Precio                        1
Descuento                 96688
dtype: int64


#**Trabajamos con la fechas para establecer los turnos del negocio con los datos presentes**

In [24]:
#Transformación de datos a datetime
#Ajustar fecha y crear columnas con Hora y DiaSemana
df["Fecha"] = pd.to_datetime(df["Fecha"], format="%Y-%m-%d %I:%M %p", errors='coerce')
#Extraer la hora real en formato 24h
df["Hora"] = df["Fecha"].dt.time
#Extraer el dia de la semana
df["DiaSemana"] = df["Fecha"].dt.day_name()
df.head()


,Fecha,Nro. del Pedido,Nombre del Producto,Variación del Producto,Cantidad,Precio,Descuento,Hora,DiaSemana
0,2022-06-10 19:22:00,1110,Shirley Temple,NaN,2.0,10.0,NaN,19:22:00,Friday
1,2022-06-10 19:23:00,1111,Freddie Mercury,NaN,1.0,25.0,NaN,19:23:00,Friday
2,2022-06-10 19:23:00,1111,Michael Jackson,NaN,1.0,20.0,NaN,19:23:00,Friday
3,2022-06-10 19:26:00,1112,Jarra Appleton,NaN,1.0,38.0,NaN,19:26:00,Friday
4,2022-06-10 19:26:00,1112,Pilsen,NaN,1.0,10.0,NaN,19:26:00,Friday


In [25]:
#Traducir los dias de la semana a español
traduccion_dias = {
    'Monday': 'Lunes', 'Tuesday': 'Martes', 'Wednesday': 'Miércoles',
    'Thursday': 'Jueves', 'Friday': 'Viernes', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
}
df["DiaSemana"] = df["Fecha"].dt.day_name().map(traduccion_dias)
df.head()

,Fecha,Nro. del Pedido,Nombre del Producto,Variación del Producto,Cantidad,Precio,Descuento,Hora,DiaSemana
0,2022-06-10 19:22:00,1110,Shirley Temple,NaN,2.0,10.0,NaN,19:22:00,Viernes
1,2022-06-10 19:23:00,1111,Freddie Mercury,NaN,1.0,25.0,NaN,19:23:00,Viernes
2,2022-06-10 19:23:00,1111,Michael Jackson,NaN,1.0,20.0,NaN,19:23:00,Viernes
3,2022-06-10 19:26:00,1112,Jarra Appleton,NaN,1.0,38.0,NaN,19:26:00,Viernes
4,2022-06-10 19:26:00,1112,Pilsen,NaN,1.0,10.0,NaN,19:26:00,Viernes


##Horarios son nocturnos V-S-D con inicio a las 6 PM

In [26]:
def fecha_negocio(row):
  if row["Fecha"].hour < 6: #antes de las 6 am = todavia es noche anterior
    return (row["Fecha"] - pd.Timedelta(days=1)).date()
  else:
    return row["Fecha"].date()

df["FechaNegocio"] = df.apply(fecha_negocio, axis=1)
df["FechaNegocio"] = pd.to_datetime(df["FechaNegocio"])
df["DiaNegocio"] = df["FechaNegocio"].dt.day_name()

#Traducir DiaNegocio a español
df["DiaNegocio"] = df["FechaNegocio"].dt.day_name().map(traduccion_dias)
df.head(10)



,Fecha,Nro. del Pedido,Nombre del Producto,Variación del Producto,Cantidad,Precio,Descuento,Hora,DiaSemana,FechaNegocio,DiaNegocio
0,2022-06-10 19:22:00,1110,Shirley Temple,NaN,2.0,10.0,NaN,19:22:00,Viernes,2022-06-10,Viernes
1,2022-06-10 19:23:00,1111,Freddie Mercury,NaN,1.0,25.0,NaN,19:23:00,Viernes,2022-06-10,Viernes
2,2022-06-10 19:23:00,1111,Michael Jackson,NaN,1.0,20.0,NaN,19:23:00,Viernes,2022-06-10,Viernes
3,2022-06-10 19:26:00,1112,Jarra Appleton,NaN,1.0,38.0,NaN,19:26:00,Viernes,2022-06-10,Viernes
4,2022-06-10 19:26:00,1112,Pilsen,NaN,1.0,10.0,NaN,19:26:00,Viernes,2022-06-10,Viernes
5,2022-06-10 19:26:00,1112,Cuzqueña Malta,NaN,1.0,10.0,NaN,19:26:00,Viernes,2022-06-10,Viernes
6,2022-06-10 19:26:00,1112,Coca Cola 500 ml,NaN,1.0,5.0,NaN,19:26:00,Viernes,2022-06-10,Viernes
7,2022-06-10 19:32:00,1113,Coca Cola 500 ml,NaN,1.0,5.0,NaN,19:32:00,Viernes,2022-06-10,Viernes
8,2022-06-10 19:37:00,1114,Cindy Lauper,NaN,1.0,13.0,NaN,19:37:00,Viernes,2022-06-10,Viernes
9,2022-06-10 19:37:00,1114,Bon Jovi,NaN,1.0,15.0,NaN,19:37:00,Viernes,2022-06-10,Viernes


##Revision que las fechas correspondan realmente al turno V-S-D

###Nota: Negocio tambien atiende en feriados relacionados a festividades

In [27]:
#VERIFICACIÓN DE QUE LAS MADRUGADAS CORRESPONDAN AL TURNO NOCHE
# Filtra las filas donde la hora original sea menor a 6
madrugadas_df = df[df["Fecha"].dt.hour < 6]

# Mostramos las columnas clave para que compruebes el cambio de día
madrugadas_df[["Fecha", "Hora", "DiaSemana", "FechaNegocio", "DiaNegocio"]].head(10)

,Fecha,Hora,DiaSemana,FechaNegocio,DiaNegocio
35,2022-06-11 00:30:00,00:30:00,Sábado,2022-06-10,Viernes
36,2022-06-11 00:54:00,00:54:00,Sábado,2022-06-10,Viernes
75,2022-06-18 00:16:00,00:16:00,Sábado,2022-06-17,Viernes
76,2022-06-18 00:19:00,00:19:00,Sábado,2022-06-17,Viernes
77,2022-06-18 00:29:00,00:29:00,Sábado,2022-06-17,Viernes
78,2022-06-18 00:29:00,00:29:00,Sábado,2022-06-17,Viernes
79,2022-06-18 00:42:00,00:42:00,Sábado,2022-06-17,Viernes
80,2022-06-18 01:00:00,01:00:00,Sábado,2022-06-17,Viernes
81,2022-06-18 01:05:00,01:05:00,Sábado,2022-06-17,Viernes
82,2022-06-18 01:18:00,01:18:00,Sábado,2022-06-17,Viernes


#Tratamiento de Nulos

In [28]:
df["Descuento"] = df["Descuento"].fillna(0)
df["Variación del Producto"] = df["Variación del Producto"].fillna("Sin variación")
df.head()

,Fecha,Nro. del Pedido,Nombre del Producto,Variación del Producto,Cantidad,Precio,Descuento,Hora,DiaSemana,FechaNegocio,DiaNegocio
0,2022-06-10 19:22:00,1110,Shirley Temple,Sin variación,2.0,10.0,0.0,19:22:00,Viernes,2022-06-10,Viernes
1,2022-06-10 19:23:00,1111,Freddie Mercury,Sin variación,1.0,25.0,0.0,19:23:00,Viernes,2022-06-10,Viernes
2,2022-06-10 19:23:00,1111,Michael Jackson,Sin variación,1.0,20.0,0.0,19:23:00,Viernes,2022-06-10,Viernes
3,2022-06-10 19:26:00,1112,Jarra Appleton,Sin variación,1.0,38.0,0.0,19:26:00,Viernes,2022-06-10,Viernes
4,2022-06-10 19:26:00,1112,Pilsen,Sin variación,1.0,10.0,0.0,19:26:00,Viernes,2022-06-10,Viernes


In [29]:
df[df["Nombre del Producto"].isnull()]
df = df.dropna(subset=["Nombre del Producto", "Cantidad", "Precio"])
df.shape

(96687, 11)

#Ver rango de fechas

In [30]:
df["Fecha"].min()


Timestamp('2022-06-10 19:22:00')

In [31]:
df["Fecha"].max()

Timestamp('2026-08-03 00:13:00')

#Separación de producto real y meseros en columna "Nombre de Producto"

In [32]:
#Filas de meseros: Precio = 0 y sin variación
es_mesero = (df["Precio"]== 0) & (df["Variación del Producto"] == "Sin variación")
#Verificar que realmente sean nombres de personas
print(df[es_mesero]["Nombre del Producto"].unique())

['Lisandro' 'Melisa' 'Ivan' 'Kathy Marjorie' 'Sra. Irma' 'Diana' 'Barra'
 'Melissa' 'Mary-Barra' 'Luis' 'Mary Luz' 'Gris' 'Bryan' 'Piero' 'George'
 'Rendel' 'Andree' 'Alex' 'Jackeline' 'Noelia' 'Jesus' 'Karina' 'Mire'
 'Jacky' 'Pilar' 'Ronaldo' 'Yamilet' 'Marcia' 'Yessica' 'Brenda'
 'Estrella' 'Lisbeth' 'Carlos' 'NN2' 'Jackie' 'Elena' 'Jorge' 'Marcia 2'
 'Jose' 'NNN' 'Dayana' 'Manuel' 'Angie' 'NN' 'Cesar' 'Jean Piere'
 'Elizabeth' 'David' 'Jana' 'Milagros' 'Silvana' 'Sasha' 'André' 'Nn'
 'Fernando' 'Jhoana' 'Luz' 'Joseph' 'Anthony' 'Anabel' 'Josue' 'Nicol'
 'Maria' 'Jenny' 'Mary' 'Maylu' 'Esthefani' 'Ana' 'Fernanda' 'Saúl'
 'Esaudh' 'Joselyn' 'Harold' 'Ani' 'Nadine' 'Andrea' 'Patricia' 'Jhair'
 'Cielo' 'Rocio' 'Daniela' 'Karin' 'Ronald' 'Yamileth' 'Jair' 'Rodrigo'
 'Edith']


In [33]:
# Tabla de mesera por pedido
mozas = df[es_mesero][["Nro. del Pedido", "Nombre del Producto"]].rename(
    columns={"Nombre del Producto": "Mesero"}
)

# Tabla de productos reales (sin las filas de mesera)
productos = df[~es_mesero].copy()

# Une la mesera a cada producto de su pedido
productos = productos.merge(mozas, on="Nro. del Pedido", how="left")

productos.head()

,Fecha,Nro. del Pedido,Nombre del Producto,Variación del Producto,Cantidad,Precio,Descuento,Hora,DiaSemana,FechaNegocio,DiaNegocio,Mesero
0,2022-06-10 19:22:00,1110,Shirley Temple,Sin variación,2.0,10.0,0.0,19:22:00,Viernes,2022-06-10,Viernes,NaN
1,2022-06-10 19:23:00,1111,Freddie Mercury,Sin variación,1.0,25.0,0.0,19:23:00,Viernes,2022-06-10,Viernes,NaN
2,2022-06-10 19:23:00,1111,Michael Jackson,Sin variación,1.0,20.0,0.0,19:23:00,Viernes,2022-06-10,Viernes,NaN
3,2022-06-10 19:26:00,1112,Jarra Appleton,Sin variación,1.0,38.0,0.0,19:26:00,Viernes,2022-06-10,Viernes,NaN
4,2022-06-10 19:26:00,1112,Pilsen,Sin variación,1.0,10.0,0.0,19:26:00,Viernes,2022-06-10,Viernes,NaN


#Tratamiento de nombres, sus variantes y duplicados en tabla productos

Nota: Segun información del negocio los nombres de meseros NN', 'NN2', 'NNN', 'Nn', 'nN', 'Nn2', 'Mary-Barra' corresponden a pedidos en Barra

Hay que convertir todos estos nombres a Barra

In [34]:
#Definimos los nombres incorrectos y sus variantes que significan "Barra"
variantes_barra = ['NN', 'NN2', 'NNN', 'Nn', 'nN', 'Nn2', 'Mary-Barra']
#Reemplazamos todas esas variantes por el nombre único 'Barra'
productos['Mesero'] = productos['Mesero'].replace(variantes_barra, 'Barra')
#Verificamos el cambio de nombre a Barra
print(productos['Mesero'].unique())

[nan 'Lisandro' 'Melisa' 'Ivan' 'Kathy Marjorie' 'Sra. Irma' 'Diana'
 'Barra' 'Melissa' 'Mary Luz' 'Luis' 'Gris' 'Bryan' 'Piero' 'George'
 'Rendel' 'Andree' 'Alex' 'Noelia' 'Jackeline' 'Jesus' 'Karina' 'Mire'
 'Jacky' 'Pilar' 'Ronaldo' 'Yamilet' 'Marcia' 'Yessica' 'Brenda'
 'Estrella' 'Lisbeth' 'Carlos' 'Jackie' 'Elena' 'Jorge' 'Marcia 2' 'Jose'
 'Dayana' 'Manuel' 'Angie' 'Cesar' 'Jean Piere' 'Elizabeth' 'David' 'Jana'
 'Milagros' 'Silvana' 'Sasha' 'André' 'Fernando' 'Jhoana' 'Luz' 'Joseph'
 'Anthony' 'Anabel' 'Josue' 'Nicol' 'Maria' 'Jenny' 'Mary' 'Maylu'
 'Esthefani' 'Ana' 'Fernanda' 'Saúl' 'Esaudh' 'Joselyn' 'Harold' 'Ani'
 'Nadine' 'Andrea' 'Patricia' 'Jhair' 'Cielo' 'Rocio' 'Daniela' 'Karin'
 'Ronald' 'Yamileth' 'Jair' 'Rodrigo' 'Edith']


### Se hizo el hallazgo de valores NaN en los meseros, para lo cual se tuvo la respuesta del Negocio que corresponden a pedidos donde el Mesero no se registro.

In [35]:
#Identificar NaN en meseros
print(productos.isna().sum())

Fecha                        0
Nro. del Pedido              0
Nombre del Producto          0
Variación del Producto       0
Cantidad                     0
Precio                       0
Descuento                    0
Hora                         0
DiaSemana                    0
FechaNegocio                 0
DiaNegocio                   0
Mesero                    1023
dtype: int64


In [36]:
#Tratamiento de NaN en pedidos no registrados por el mesero
productos["Mesero"] = productos["Mesero"].fillna("Sin registrar")
#Verificar
print(productos.isna().sum())

Fecha                     0
Nro. del Pedido           0
Nombre del Producto       0
Variación del Producto    0
Cantidad                  0
Precio                    0
Descuento                 0
Hora                      0
DiaSemana                 0
FechaNegocio              0
DiaNegocio                0
Mesero                    0
dtype: int64


###Otro hallazgo es el duplicado de nombres, el negocio aclaro que se trataba de la misma persona, se hizo dicha modificación cuando retornaron a laborar despues de un periodo

In [37]:
### Consolidación de nombres de personal (Barra + duplicados)
variantes_barra = ['NN', 'NN2', 'NNN', 'Nn', 'nN', 'Nn2', 'Mary-Barra']
productos['Mesero'] = productos['Mesero'].replace(variantes_barra, 'Barra')

variantes_nombres = {
    "Melissa": "Melisa",
    "Marcia 2": "Marcia",
}
productos["Mesero"] = productos["Mesero"].replace(variantes_nombres)

### Anonimización de nombres — antes de cualquier cálculo de negocio
meseros_unicos = productos["Mesero"].unique()
mapeo_anonimo = {nombre: f"Mesero_{i+1}" for i, nombre in enumerate(sorted(meseros_unicos)) if nombre != "Sin registrar"}
mapeo_anonimo["Sin registrar"] = "Sin registrar"

# Sobrescribe la columna Mesero directamente con el alias — ya no existe
# ninguna versión con nombres reales en el DataFrame de aquí en adelante
productos["Mesero"] = productos["Mesero"].map(mapeo_anonimo)
print(productos["Mesero"].unique())

['Sin registrar' 'Mesero_50' 'Mesero_60' 'Mesero_31' 'Mesero_49'
 'Mesero_78' 'Mesero_19' 'Mesero_10' 'Mesero_58' 'Mesero_52' 'Mesero_29'
 'Mesero_12' 'Mesero_67' 'Mesero_28' 'Mesero_69' 'Mesero_5' 'Mesero_1'
 'Mesero_65' 'Mesero_32' 'Mesero_39' 'Mesero_48' 'Mesero_62' 'Mesero_34'
 'Mesero_68' 'Mesero_73' 'Mesero_79' 'Mesero_55' 'Mesero_81' 'Mesero_11'
 'Mesero_25' 'Mesero_51' 'Mesero_13' 'Mesero_33' 'Mesero_21' 'Mesero_42'
 'Mesero_43' 'Mesero_18' 'Mesero_54' 'Mesero_7' 'Mesero_14' 'Mesero_37'
 'Mesero_22' 'Mesero_17' 'Mesero_36' 'Mesero_61' 'Mesero_76' 'Mesero_74'
 'Mesero_6' 'Mesero_27' 'Mesero_41' 'Mesero_53' 'Mesero_45' 'Mesero_9'
 'Mesero_3' 'Mesero_46' 'Mesero_64' 'Mesero_56' 'Mesero_38' 'Mesero_57'
 'Mesero_59' 'Mesero_24' 'Mesero_2' 'Mesero_26' 'Mesero_75' 'Mesero_23'
 'Mesero_44' 'Mesero_30' 'Mesero_8' 'Mesero_63' 'Mesero_4' 'Mesero_66'
 'Mesero_40' 'Mesero_15' 'Mesero_70' 'Mesero_16' 'Mesero_47' 'Mesero_72'
 'Mesero_80' 'Mesero_35' 'Mesero_71' 'Mesero_20']


#Nota: a la fecha presente Agosto 2026 no tiene los suficientes datos para considerarlo en el analisis, solo posee 1 y 2 de agosto.

###Por lo cual no lo consideraremos en la tabla producto

In [38]:
# Eliminamos de raíz cualquier registro que pertenezca a Agosto 2026 en adelante
productos = productos[productos["FechaNegocio"] < "2026-08-01"]

In [39]:
productos.to_csv("productos_clean.csv", index=False)